# Argos — Visualization of Forensic Findings
Graphical analysis of contract fragmentation in Chilean public procurement, 2025.  
Source: Mercado Público · Neo4j Gold Layer.

> **Methodological note:** This analysis detects statistical indicators. It does not constitute proof of fraud. It requires independent documentary verification.

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
from neo4j import GraphDatabase
from dotenv import load_dotenv

warnings.filterwarnings('ignore')
load_dotenv('../.env')

driver = GraphDatabase.driver(
    'bolt://localhost:7687',
    auth=('neo4j', os.getenv('NEO4J_ROOT_PASSWORD'))
)

def run_query(query, params=None):
    """Utility function to return Cypher results as a Pandas DataFrame."""
    with driver.session() as session:
        result = session.run(query, params or {})
        return pd.DataFrame([dict(r) for r in result])

ORDER_LIMIT   = 6_600_000
TENDER_LIMIT = 66_000_000

C = {
    'red':    '#DC2626',
    'orange': '#D97706',
    'blue':   '#2563EB',
    'green':  '#16A34A',
    'gray':   '#6B7280',
    'bg':     '#F9FAFB',
    'border': '#E5E7EB',
}

def fmt_clp(x, _=None):
    if x >= 1e12: return f'{x/1e12:.1f} T'
    if x >= 1e9:  return f'{x/1e9:.1f} B'
    if x >= 1e6:  return f'{x/1e6:.0f} M'
    return f'{x:,.0f}'

def apply_style(ax, title, xlabel='', ylabel='', grid='x'):
    ax.set_facecolor(C['bg'])
    ax.figure.patch.set_facecolor('white')
    ax.set_title(title, fontsize=13, fontweight='bold', pad=12, loc='left')
    ax.set_xlabel(xlabel, fontsize=10)
    ax.set_ylabel(ylabel, fontsize=10)
    ax.tick_params(labelsize=9)
    ax.spines[['top', 'right']].set_visible(False)
    ax.spines[['left', 'bottom']].set_color(C['border'])
    ax.grid(axis=grid, color=C['border'], linewidth=0.7, linestyle='--')
    ax.set_axisbelow(True)

os.makedirs('../docs/img', exist_ok=True)
print('OK')

---
## Chart 1 — Graph Scale

How many nodes of each type are in the database.

In [ ]:
df_stats = run_query("""
    MATCH (n)
    RETURN labels(n)[0] AS type, count(n) AS total
    ORDER BY total DESC
""")

from IPython.display import display
display(df_stats)

---
## Chart 2 — Top 15 Fragmentation

The Agency-Vendor pairs with the highest monthly accumulation under the Compra Ágil modality, ordered by total amount. 
Vertical lines mark the legal thresholds.

In [ ]:
df_frag = run_query("""
    MATCH (u:UnidadCompra)-[:EMITIO]->(oc:OrdenCompra_Item {Modalidad: 'AG'})-[:ADJUDICADA_A]->(p:Proveedor)
    WITH u.UnidadCompra AS agency,
         p.Nombre       AS vendor,
         oc.Fecha.month AS month,
         count(oc)      AS order_count,
         sum(oc.Monto)  AS total_clp
    WHERE total_clp > $threshold
    RETURN agency, vendor, month, order_count, total_clp
    ORDER BY total_clp DESC
    LIMIT 15
""", {'threshold': TENDER_LIMIT})

print(f"{len(df_frag)} Agency-Vendor pairs in the top 15.")

fig, ax = plt.subplots(figsize=(10, 6))

y_labels = [f"{r['vendor'][:30]}...\n({r['agency'][:30]}...)" for _, r in df_frag.iterrows()]
bars = ax.barh(y_labels, df_frag['total_clp'], color=C['blue'], alpha=0.8)

ax.invert_yaxis()
ax.xaxis.set_major_formatter(mticker.FuncFormatter(fmt_clp))

# Thresholds
ax.axvline(TENDER_LIMIT, color=C['red'], linestyle='--', linewidth=1.2, label='Tender Threshold (1,000 UTM)')

apply_style(ax, "Top 15 Fragmentation Clusters", "Total Monthly Amount (CLP)", "")
ax.legend(frameon=False, loc='lower right')

plt.tight_layout()
plt.savefig('../docs/img/top_frag.png', dpi=300)
plt.show()

---
## Chart 3 — Distribution of Suspicion Levels

Summary of the automatic forensic classification.

In [ ]:
import re
from collections import Counter

print("Querying Neo4j...")
df_raw = run_query("""
    MATCH (u:UnidadCompra)-[:EMITIO]->(oc:OrdenCompra_Item {Modalidad: 'AG'})-[:ADJUDICADA_A]->(p:Proveedor)
    WITH u.UnidadCompra AS agency,
         p.Nombre        AS vendor,
         oc.Fecha.month  AS month,
         count(oc)                    AS order_count,
         sum(oc.Monto)                AS total_clp,
         collect(oc.Monto)            AS amounts,
         collect(oc.NombreEspecifico) AS descriptions,
         collect(oc.CodigoUnico)      AS ids
    WHERE order_count > 1 AND total_clp >= $limit
    RETURN agency, vendor, month, order_count, total_clp, amounts, descriptions, ids
""", {'limit': float(TENDER_LIMIT)})

def analyze_case(row):
    amounts = [float(m) for m in row['amounts'] if m is not None]
    descs   = [str(d) for d in row['descriptions'] if d is not None]
    ids     = [str(i) for i in row['ids'] if i is not None]
    n       = len(amounts)
    if n == 0: return 'LOW'

    pct_identical_amount = Counter(amounts).most_common(1)[0][1] / n * 100
    pct_identical_desc   = Counter(descs).most_common(1)[0][1] / n * 100
    unique_quotes       = len(set([re.match(r'(\d+-\d+-AG25)', i).group(1) for i in ids if re.match(r'(\d+-\d+-AG25)', i)]))

    if pct_identical_amount > 80 and pct_identical_desc > 80: return 'HIGH'
    if pct_identical_amount > 80 or pct_identical_desc > 80 or unique_quotes == 1: return 'MEDIUM'
    return 'LOW'

df_raw['suspicion'] = df_raw.apply(analyze_case, axis=1)
counts = df_raw['suspicion'].value_counts()

fig, ax = plt.subplots(figsize=(7, 4))
colors = {'HIGH': C['red'], 'MEDIUM': C['orange'], 'LOW': C['green']}
order = ['HIGH', 'MEDIUM', 'LOW']
bars = ax.bar(order, [counts.get(o, 0) for o in order], color=[colors[o] for o in order])

for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, f'{int(bar.get_height())}', 
            ha='center', va='bottom', fontweight='bold')

apply_style(ax, "Distribution of Suspicion Levels", "", "Number of Cases", grid='y')
plt.tight_layout()
plt.savefig('../docs/img/suspicion_dist.png', dpi=300)
plt.show()

---
## Final Classification — with Chronicity and Legal Basis

Incorporates three independent dimensions:
1. **Forensic Evidence** — identical amounts/descriptions, base quote
2. **Chronicity** — how many months of the year the same pair repeated
3. **Scale** — multiplier over the tender threshold

Legal Basis: DS 250 art. 13 and DS 661 art. 16 (December 2024). CGR ruling 
that rejected the argument of "independent cost centers" when the total exceeded 1,000 UTM.

In [ ]:
def compute_indicators(row):
    """Extract forensic indicators from the raw Neo4j data row."""
    montos = [float(m) for m in row['amounts'] if m is not None]
    descs  = [str(d) for d in row['descriptions'] if d is not None]
    ids    = [str(i) for i in row['ids'] if i is not None]
    n = len(montos)

    if n == 0:
        return pd.Series({'Monto_Min': 0, 'Monto_Max': 0, 'Monto_Medio': 0,
                          'Ordenes_Sobre_Limite': 0, 'Pct_Monto_Identico': 0,
                          'Pct_Desc_Identica': 0, 'Cotizacion_Base': '—',
                          'Cotizaciones_Unicas': 0, 'Desc_Muestra': '—'})

    monto_top = Counter(montos).most_common(1)[0][1]
    desc_top  = Counter(descs).most_common(1)[0][1] if descs else 0

    bases = [re.match(r"(\d+-\d+-AG25)", i).group(1)
             for i in ids if re.match(r"(\d+-\d+-AG25)", i)]
    cot_unicas = len(set(bases))
    cot_base   = Counter(bases).most_common(1)[0][0] if bases else '—'

    return pd.Series({
        'Monto_Min':            min(montos),
        'Monto_Max':            max(montos),
        'Monto_Medio':          sum(montos) / n,
        'Ordenes_Sobre_Limite': sum(1 for m in montos if m > ORDER_LIMIT),
        'Pct_Monto_Identico':   round(monto_top / n * 100, 1),
        'Pct_Desc_Identica':    round(desc_top  / n * 100, 1),
        'Cotizacion_Base':      cot_base,
        'Cotizaciones_Unicas':  cot_unicas,
        'Desc_Muestra':         descs[0][:120] if descs else '—',
    })

print("Computing indicators...")
indicators = df_raw.apply(compute_indicators, axis=1)
df = pd.concat([df_raw.drop(columns=['amounts', 'descriptions', 'ids']), indicators], axis=1)
df['Multiplier'] = (df['total_clp'] / TENDER_LIMIT).round(1)
df['Month_Name'] = df['month'].map({1:"Jan",2:"Feb",3:"Mar",4:"Apr",5:"May",6:"Jun",
                                     7:"Jul",8:"Aug",9:"Sep",10:"Oct",11:"Nov",12:"Dec"})

# Chronicity calculation
chronicity = (df.groupby(['agency', 'vendor'])
               .agg(Month_Count=('month', 'count'),
                    Month_List=('Month_Name', lambda x: sorted(set(x))),
                    Annual_Total=('total_clp', 'sum'))
               .reset_index())
df = df.merge(chronicity, on=['agency', 'vendor'], how='left')

# Scoring system
def score_evidence(row):
    if row['Pct_Monto_Identico'] > 80 and row['Pct_Desc_Identica'] > 80: return 3
    if row['Cotizaciones_Unicas'] == 1: return 3
    if row['Pct_Monto_Identico'] > 80 or row['Pct_Desc_Identica'] > 80: return 2
    return 1

def score_chronicity(n):
    if n >= 6: return 3
    if n >= 3: return 2
    return 1

def score_scale(mult):
    if mult >= 50: return 3
    if mult >= 10: return 2
    return 1

df['Score_Evidence']   = df.apply(score_evidence, axis=1)
df['Score_Chronicity'] = df['Month_Count'].apply(score_chronicity)
df['Score_Scale']      = df['Multiplier'].apply(score_scale)
df['Score_Total']      = df['Score_Evidence'] + df['Score_Chronicity'] + df['Score_Scale']

def get_risk_level(row):
    s = row['Score_Total']
    if s >= 7: return 'CRÍTICO'
    if s >= 6: return 'ALTO'
    if s >= 4: return 'MEDIO'
    return 'BAJO'

df['Final_Level'] = df.apply(get_risk_level, axis=1)
df.head(3)

In [ ]:
# Legal argument generation per case (Keeping Spanish content for report output)
SPANISH_MONTHS = {1:"enero",2:"febrero",3:"marzo",4:"abril",5:"mayo",6:"junio",
                  7:"julio",8:"agosto",9:"septiembre",10:"octubre",11:"noviembre",12:"diciembre"}

def get_legal_basis(row):
    parts = []
    if row['Score_Evidence'] == 3:
        parts.append(
            f"Órdenes con {row['Pct_Monto_Identico']:.0f}% monto idéntico y "
            f"{row['Pct_Desc_Identica']:.0f}% descripción idéntica, "
            f"derivadas de {row['Cotizaciones_Unicas']} cotización/es base."
        )
    elif row['Score_Evidence'] == 2:
        parts.append("Montos o descripciones parcialmente repetitivos.")
    else:
        parts.append("Variedad en montos y descripciones — compras diversas posibles.")

    if row['Score_Chronicity'] == 3:
        parts.append(
            f"Patrón sistemático en {row['Month_Count']} meses. "
            f"La recurrencia elimina el argumento de imprevisibilidad "
            f"(CGR, Municipalidad de La Cisterna)."
        )
    elif row['Score_Chronicity'] == 2:
        parts.append(f"Patrón recurrente en {row['Month_Count']} meses.")

    if row['Score_Scale'] >= 2:
        parts.append(
            f"Total acumulado: ${row['total_clp']/1e6:,.0f} M CLP "
            f"({row['Multiplier']:.1f}x el umbral de licitación)."
        )
    return " ".join(parts)

df['Legal_Basis'] = df.apply(get_legal_basis, axis=1)

# Export to forensic report format for Chile
def case_summary_md(r, idx):
    pct = r['Monto_Medio'] / ORDER_LIMIT * 100
    month_str = SPANISH_MONTHS.get(int(r['month']), str(r['month']))
    return "\n".join([
        f"### {idx}. {r['agency']}",
        "",
        "| | |",
        "|---|---|",
        f"| Proveedor | `{r['vendor']}` |",
        f"| Mes | {month_str} 2025 |",
        f"| Órdenes | **{int(r['order_count'])}** |",
        f"| Total acumulado | **${r['total_clp']/1e6:,.1f} M CLP** |",
        f"| Multiplicador | **{r['Multiplier']:.1f}x** el umbral de licitación |",
        f"| % monto idéntico | {r['Pct_Monto_Identico']:.0f}% |",
        f"| % descripción idéntica | {r['Pct_Desc_Identica']:.0f}% |",
        f"| Nivel | **{r['Final_Level']}** |",
        "",
        f"**Descripción:** _{r['Desc_Muestra']}_",
        "",
        f"> {r['Legal_Basis']}",
        "---",
    ])

os.makedirs('../docs', exist_ok=True)
report_path = '../docs/forensic_report_final.md'
with open(report_path, 'w', encoding='utf-8') as f:
    f.write("# Forensic Report — Fragmentation in Chilean Public Procurement 2025\n\n")
    # Export top 10 suspicious cases
    top_cases = df.sort_values('Score_Total', ascending=False).head(10)
    for i, (_, row) in enumerate(top_cases.iterrows(), 1):
        f.write(case_summary_md(row, i))

print(f"Forensic report exported to: {report_path}")
driver.close()
print('Connection closed.')